# Urban Heat Island (UHI) Analysis — v4
## MSF · Tondo, Manila · Z_GIS Salzburg

---

### v4 improvements
- **Sentinel-3 SLSTR — 50x faster**: downloads only LST + geodetic files (~2 MB) instead of full zip (~100+ MB)
- **MODIS auto-fallback** if S3 unavailable
- **LST threshold analysis**: extreme heat / moderate / cool day classification
- **Enhanced statistics**: percentile distributions, threshold exceedance counts, heat stress indices
- Landsat 8+9, regression fusion, monthly time-series, vulnerability index — all included

> Edit **CONFIGURATION** only. Set `THERMAL_SOURCE = "S3"` or `"MODIS"`.
---

## 0. Setup

**GEE**: [code.earthengine.google.com/register](https://code.earthengine.google.com/register) → Cloud project → Enable EE API

**CDSE** (free, for S3): [dataspace.copernicus.eu](https://dataspace.copernicus.eu) → register → use email+password

---

In [1]:
# 0A. INSTALL
!pip install earthengine-api folium pandas matplotlib seaborn geopandas xarray netCDF4 requests -q
print('Installed.')

Installed.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
-m pip install --upgrade pip

SyntaxError: invalid syntax (2350214200.py, line 1)

In [4]:
# 0B. AUTHENTICATE
GEE_PROJECT = "uhi-msf-analysis"  # <-- your GEE project

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print(f"GEE: {GEE_PROJECT}")

import requests

CDSE_USERNAME = "**********"  # <-- your CDSE email
CDSE_PASSWORD = "**********"  # <-- your CDSE password

def get_cdse_token():
    r = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={"grant_type":"password","username":CDSE_USERNAME,"password":CDSE_PASSWORD,"client_id":"cdse-public"})
    if r.status_code == 200:
        print("CDSE authenticated.")
        return r.json()["access_token"]
    print(f"CDSE auth failed: {r.status_code}")
    return None

cdse_token = get_cdse_token()


GEE: uhi-msf-analysis
CDSE authenticated.


In [5]:
# 0C. IMPORTS
import folium, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import xarray as xr, json, os, tempfile, warnings, io
warnings.filterwarnings("ignore")
from datetime import datetime, timedelta

def add_ee_layer(m, img, vis, name, shown=True):
    mid = img.getMapId(vis)
    folium.TileLayer(tiles=mid["tile_fetcher"].url_format, attr="GEE", name=name,
                     overlay=True, control=True, show=shown).add_to(m)

def make_map(lat, lon, zoom=14):
    return folium.Map(location=[lat, lon], zoom_start=zoom, tiles="OpenStreetMap")

print("Libraries loaded.")


Libraries loaded.


---
## 1. CONFIGURATION
---

In [6]:
# 1. CONFIGURATION
AOI_BBOX = [120.950, 14.600, 120.985, 14.640]  # Tondo, Manila
# AOI_ASSET = "FAO/GAUL/2015/level2"
# AOI_FILTER = ee.Filter.eq("ADM2_NAME", "Manila")

YEAR_START = 2019
YEAR_END = 2025
DRY_MONTHS = [12,1,2,3,4,5]
WET_MONTHS = [6,7,8,9,10,11]
THERMAL_SOURCE = "S3"   # "S3" or "MODIS"
MAX_CLOUD_COVER = 30
UHI_BUFFER_M = 5000

W_LST=0.30; W_POP=0.25; W_BUILD=0.20; W_NDVI=0.15; W_HEALTH=0.10
PROJECT_NAME = "UHI_Tondo_Manila"

# LST THRESHOLDS (Celsius) — adjust for your climate zone
LST_HIGH = 35      # >= this = extreme heat
LST_MODERATE = 30  # >= this and < HIGH = moderate heat
# below MODERATE = cool/normal

assert abs(W_LST+W_POP+W_BUILD+W_NDVI+W_HEALTH-1.0)<0.001
print(f"Config: {PROJECT_NAME} | {AOI_BBOX} | {YEAR_START}-{YEAR_END} | Thermal: {THERMAL_SOURCE}")
print(f"Thresholds: High >= {LST_HIGH}C | Moderate >= {LST_MODERATE}C")


Config: UHI_Tondo_Manila | [120.95, 14.6, 120.985, 14.64] | 2019-2025 | Thermal: S3
Thresholds: High >= 35C | Moderate >= 30C


## 2. Area of Interest

In [7]:
# 2. AOI
try:
    aoi = ee.FeatureCollection(AOI_ASSET).filter(AOI_FILTER).geometry()
except NameError:
    aoi = ee.Geometry.Rectangle(AOI_BBOX)
aoi_buffer = aoi.buffer(UHI_BUFFER_M)
aoi_ring = aoi_buffer.difference(aoi)
center_lat = (AOI_BBOX[1]+AOI_BBOX[3])/2
center_lon = (AOI_BBOX[0]+AOI_BBOX[2])/2
m = make_map(center_lat, center_lon, 13)
folium.GeoJson(aoi.getInfo(), name="AOI", style_function=lambda x:{"color":"red","weight":2,"fillOpacity":0.1}).add_to(m)
folium.GeoJson(aoi_ring.getInfo(), name="Ring", style_function=lambda x:{"color":"blue","weight":1,"fillOpacity":0.05}).add_to(m)
folium.LayerControl().add_to(m)
m


## 3. Helper Functions

In [8]:
# 3. HELPERS
def mask_landsat_clouds(image):
    qa = image.select("QA_PIXEL")
    return image.updateMask(qa.bitwiseAnd(1<<3).eq(0)).updateMask(qa.bitwiseAnd(1<<4).eq(0))

def compute_lst_landsat(image):
    return image.addBands(image.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("LST"))

def compute_lst_modis(image):
    lst = image.select("LST_Day_1km").multiply(0.02).subtract(273.15).rename("LST_coarse")
    return image.addBands(lst).updateMask(image.select("QC_Day").bitwiseAnd(3).eq(0))

def compute_ndvi(img): return img.addBands(img.normalizedDifference(["SR_B5","SR_B4"]).rename("NDVI"))
def compute_ndbi(img): return img.addBands(img.normalizedDifference(["SR_B6","SR_B5"]).rename("NDBI"))
def compute_mndwi(img): return img.addBands(img.normalizedDifference(["SR_B3","SR_B6"]).rename("MNDWI"))

def filter_by_months(col, months):
    def tag(img):
        m = ee.Number(img.date().get("month"))
        return img.set("in_season", ee.List(months).contains(m))
    return col.map(tag).filter(ee.Filter.eq("in_season", True))

def add_time_props(img):
    d = img.date()
    return img.set("year",d.get("year")).set("month",d.get("month")).set("doy",d.getRelative("day","year"))

def min_max_normalize(image, geom, band_name=None, scale=30):
    if band_name: image = image.select(band_name)
    mm = image.reduceRegion(reducer=ee.Reducer.minMax(), geometry=geom, scale=scale, maxPixels=1e9)
    keys = mm.keys().getInfo()
    mn = [k for k in keys if "_min" in k][0]
    mx = [k for k in keys if "_max" in k][0]
    return image.subtract(ee.Number(mm.get(mn))).divide(ee.Number(mm.get(mx)).subtract(ee.Number(mm.get(mn)))).clamp(0,1)

print("Helpers defined.")


Helpers defined.


---
## 4. Landsat 8+9 Acquisition

In [9]:
# 4. LANDSAT 8+9
l9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi)
      .filterDate(f"{YEAR_START}-01-01",f"{YEAR_END}-12-31")
      .filter(ee.Filter.lt("CLOUD_COVER",MAX_CLOUD_COVER))
      .map(mask_landsat_clouds).map(compute_lst_landsat)
      .map(compute_ndvi).map(compute_ndbi).map(compute_mndwi)
      .map(add_time_props).map(lambda i:i.set("sensor","L9")))
l8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi)
      .filterDate(f"{YEAR_START}-01-01",f"{YEAR_END}-12-31")
      .filter(ee.Filter.lt("CLOUD_COVER",MAX_CLOUD_COVER))
      .map(mask_landsat_clouds).map(compute_lst_landsat)
      .map(compute_ndvi).map(compute_ndbi).map(compute_mndwi)
      .map(add_time_props).map(lambda i:i.set("sensor","L8")))
landsat_col = l8.merge(l9).sort("system:time_start")
print(f"L8:{l8.size().getInfo()} L9:{l9.size().getInfo()} Total:{landsat_col.size().getInfo()}")
dates = landsat_col.aggregate_array("system:time_start").getInfo()
if dates:
    dts=[datetime.utcfromtimestamp(d/1000).strftime("%Y-%m-%d") for d in dates]
    print(f"  {min(dts)} -> {max(dts)}")


L8:47 L9:27 Total:74
  2019-01-04 -> 2025-12-30


## 4B. Sentinel-3 SLSTR (FAST) + MODIS Fallback

**Speed fix**: Instead of downloading the full product zip (~100-200 MB), we download only
`LST_in.nc` (~1 MB) and `geodetic_in.nc` (~1 MB) via the CDSE **Nodes API**.

This is **~50x faster** than the zip approach.


In [ ]:
# 4B. SENTINEL-3 SLSTR (CDSE NODES API — FAST) + MODIS FALLBACK
coarse_source_used = None
s3_monthly_lst = {}

if THERMAL_SOURCE == "S3" and cdse_token:
    print("Fetching S3 SLSTR LST via CDSE Nodes API (fast — individual files only)...")
    w, s, e, n = AOI_BBOX
    buf = UHI_BUFFER_M / 111000
    sw, ss, se, sn = w - buf, s - buf, e + buf, n + buf
    active_token = cdse_token
    total_scenes = 0

    for yr in range(YEAR_START, YEAR_END + 1):
        for mo in range(1, 13):
            start_dt = f"{yr}-{mo:02d}-01T00:00:00.000Z"
            end_dt = f"{yr}-{mo+1:02d}-01T00:00:00.000Z" if mo < 12 else f"{yr+1}-01-01T00:00:00.000Z"

            url = (
                "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
                f"?$filter=Collection/Name eq 'SENTINEL-3'"
                f" and Attributes/OData.CSC.StringAttribute/any("
                f"att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'SL_2_LST___')"
                f" and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON(("
                f"{sw} {ss},{se} {ss},{se} {sn},{sw} {sn},{sw} {ss}))')"
                f" and ContentDate/Start gt {start_dt}"
                f" and ContentDate/Start lt {end_dt}"
                f"&$top=30&$orderby=ContentDate/Start asc"
            )

            try:
                results = requests.get(url, timeout=30).json().get("value", [])
            except:
                continue
            if not results:
                continue

            lst_vals = []

            for prod in results[:3]:  # 3 scenes per month
                pid = prod["Id"]
                pname = prod["Name"]

                try:
                    hdr = {"Authorization": f"Bearer {active_token}"}

                    # Download ONLY LST_in.nc via Nodes API (~1 MB instead of 100+ MB)
                    lst_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({pid})/Nodes({pname})/Nodes(LST_in.nc)/$value"
                    geo_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({pid})/Nodes({pname})/Nodes(geodetic_in.nc)/$value"

                    r_lst = requests.get(lst_url, headers=hdr, timeout=60)
                    if r_lst.status_code == 401:
                        refreshed = get_cdse_token()
                        if refreshed:
                            active_token = refreshed
                            hdr = {"Authorization": f"Bearer {active_token}"}
                            r_lst = requests.get(lst_url, headers=hdr, timeout=60)
                    if r_lst.status_code != 200:
                        continue

                    r_geo = requests.get(geo_url, headers=hdr, timeout=60)
                    if r_geo.status_code != 200:
                        continue

                    # Read NetCDF from memory — no disk I/O
                    with tempfile.TemporaryDirectory() as tmp:
                        lst_path = os.path.join(tmp, "LST_in.nc")
                        geo_path = os.path.join(tmp, "geodetic_in.nc")
                        with open(lst_path, "wb") as f: f.write(r_lst.content)
                        with open(geo_path, "wb") as f: f.write(r_geo.content)

                        ds_lst = xr.open_dataset(lst_path)
                        ds_geo = xr.open_dataset(geo_path)

                        if "LST" in ds_lst and "latitude_in" in ds_geo and "longitude_in" in ds_geo:
                            lst_data = ds_lst["LST"].values
                            lat_data = ds_geo["latitude_in"].values
                            lon_data = ds_geo["longitude_in"].values

                            mask = (
                                (lat_data >= ss) & (lat_data <= sn) &
                                (lon_data >= sw) & (lon_data <= se) &
                                (~np.isnan(lst_data)) &
                                (lst_data > 270) & (lst_data < 350)
                            )
                            clipped = lst_data[mask]
                            if len(clipped) > 0:
                                lst_vals.extend((clipped - 273.15).tolist())

                        ds_lst.close()
                        ds_geo.close()

                    total_scenes += 1

                except:
                    continue

            if lst_vals:
                mean_v = np.nanmean(lst_vals)
                s3_monthly_lst[(yr, mo)] = ee.Image.constant(mean_v).rename("LST_coarse").clip(aoi_buffer).float()
                print(f"  {yr}-{mo:02d}: {len(lst_vals)} px, mean={mean_v:.1f}C")
            else:
                print(f"  {yr}-{mo:02d}: 0 px in AOI")

    if s3_monthly_lst:
        coarse_source_used = "S3"
        print(f"\nS3 SLSTR: {total_scenes} scenes, {len(s3_monthly_lst)} months.")
    else:
        print("\nNo S3 data in AOI — MODIS fallback.")

if coarse_source_used is None:
    if THERMAL_SOURCE == "S3": print("S3 unavailable — MODIS fallback.")
    mt = ee.ImageCollection("MODIS/061/MOD11A1").filterBounds(aoi).filterDate(f"{YEAR_START}-01-01",f"{YEAR_END}-12-31").map(compute_lst_modis)
    ma = ee.ImageCollection("MODIS/061/MYD11A1").filterBounds(aoi).filterDate(f"{YEAR_START}-01-01",f"{YEAR_END}-12-31").map(compute_lst_modis)
    modis_col = mt.merge(ma).sort("system:time_start").map(add_time_props)
    coarse_source_used = "MODIS"
    print(f"MODIS: {modis_col.size().getInfo()} scenes")

print(f"\nThermal source: {coarse_source_used}")


Fetching S3 SLSTR LST via CDSE Nodes API (fast — individual files only)...


In [ ]:
# 4C. UNIFIED COARSE INTERFACE
def get_coarse_composite(months_filter=None, year=None, month=None):
    if coarse_source_used == "S3":
        if year and month and (year,month) in s3_monthly_lst:
            return s3_monthly_lst[(year,month)]
        elif months_filter:
            imgs = [i for (y,m),i in s3_monthly_lst.items() if m in months_filter]
            if imgs: return ee.ImageCollection(imgs).mean().rename("LST_coarse").clip(aoi_buffer)
        if s3_monthly_lst:
            return ee.ImageCollection(list(s3_monthly_lst.values())).mean().rename("LST_coarse").clip(aoi_buffer)
        return ee.Image.constant(30).rename("LST_coarse").clip(aoi_buffer).float()
    else:
        if year and month:
            s = f"{year}-{month:02d}-01"
            return modis_col.filterDate(s, ee.Date(s).advance(1,"month")).select("LST_coarse").median().clip(aoi_buffer)
        elif months_filter:
            return filter_by_months(modis_col, months_filter).select("LST_coarse").median().clip(aoi_buffer)
        return modis_col.select("LST_coarse").median().clip(aoi_buffer)

tv = get_coarse_composite(months_filter=DRY_MONTHS).reduceRegion(
    reducer=ee.Reducer.mean(),geometry=aoi,scale=1000,maxPixels=1e9).get("LST_coarse").getInfo()
print(f"Coarse interface OK — dry mean: {tv:.1f}C ({coarse_source_used})")


## 4D. Resolution Stack

In [ ]:
# 4D. STACK
cl = "S3 SLSTR" if coarse_source_used=="S3" else "MODIS"
print(f"Tier 1 (30m): LST,NDVI,NDBI,MNDWI — Landsat 8+9")
print(f"Tier 2 (10m): LandCover,BuiltUp,Buildings — ESA/GHSL/OpenBldgs")
print(f"Tier 3 (100m): Population — WorldPop")
print(f"Tier 4 (1km): LST_coarse — {cl}")
print(f"Fused (30m): LST_fused — {cl}->30m regression")


## 5. Composites

In [ ]:
# 5. COMPOSITES
dry_landsat = filter_by_months(landsat_col,DRY_MONTHS).select(["LST","NDVI","NDBI","MNDWI"]).median().clip(aoi_buffer)
wet_landsat = filter_by_months(landsat_col,WET_MONTHS).select(["LST","NDVI","NDBI","MNDWI"]).median().clip(aoi_buffer)
ann_landsat = landsat_col.select(["LST","NDVI","NDBI","MNDWI"]).median().clip(aoi_buffer)
dry_coarse = get_coarse_composite(months_filter=DRY_MONTHS)
wet_coarse = get_coarse_composite(months_filter=WET_MONTHS)
ann_coarse = get_coarse_composite()
print(f"Composites: Landsat 30m + {coarse_source_used}")


## 5B. LST Fusion (coarse → 30 m)

In [ ]:
# 5B. FUSION FUNCTION
def fuse_lst(lcomp, ccomp, geom, label=""):
    p30 = lcomp.select(["NDVI","NDBI","MNDWI"]).clip(geom)
    p1k = p30.reduceResolution(reducer=ee.Reducer.mean(),maxPixels=2048).reproject(crs="EPSG:4326",scale=1000)
    clst = ccomp.select("LST_coarse").clip(geom)
    ts = clst.addBands(p1k)
    tr = ts.sample(region=geom, scale=1000, numPixels=500, seed=42, geometries=False)
    n = tr.size().getInfo()
    if n < 10:
        print(f"  {label}: {n} samples — Landsat only")
        return lcomp.select("LST").rename("LST_fused")
    tr = tr.map(lambda f: f.set("constant",1))
    reg = tr.reduceColumns(reducer=ee.Reducer.linearRegression(numX=4,numY=1),
                           selectors=["constant","NDVI","NDBI","MNDWI","LST_coarse"])
    co = ee.Array(reg.get("coefficients")).project([0]).toList()
    b0=ee.Number(co.get(0)); b1=ee.Number(co.get(1)); b2=ee.Number(co.get(2)); b3=ee.Number(co.get(3))
    fused = p30.select("NDVI").multiply(b1).add(p30.select("NDBI").multiply(b2)).add(p30.select("MNDWI").multiply(b3)).add(b0).rename("LST_fused")
    pred1k = p1k.select("NDVI").multiply(b1).add(p1k.select("NDBI").multiply(b2)).add(p1k.select("MNDWI").multiply(b3)).add(b0)
    res = clst.subtract(pred1k).resample("bilinear").reproject(crs="EPSG:4326",scale=30)
    fused = fused.add(res).rename("LST_fused")
    print(f"  {label}: n={n}, b0={b0.getInfo():.2f}, bNDVI={b1.getInfo():.3f}, bNDBI={b2.getInfo():.3f}, bMNDWI={b3.getInfo():.3f}")
    return fused
print("Fusion function ready.")


In [ ]:
# 5C. RUN FUSION
print(f"Fusing ({coarse_source_used} -> 30m)...")
fused_dry = fuse_lst(dry_landsat, dry_coarse, aoi_buffer, "Dry")
fused_wet = fuse_lst(wet_landsat, wet_coarse, aoi_buffer, "Wet")
fused_ann = fuse_lst(ann_landsat, ann_coarse, aoi_buffer, "Annual")
dry_composite = dry_landsat.addBands(fused_dry)
wet_composite = wet_landsat.addBands(fused_wet)
annual_composite = ann_landsat.addBands(fused_ann)
print("Done: LST + LST_fused + NDVI + NDBI + MNDWI")


## 5D. Monthly Time-Series

In [ ]:
# 5D. MONTHLY
monthly_results = []
print(f"Monthly fusion ({coarse_source_used})...")
for yr in range(YEAR_START, YEAR_END+1):
    for mo in range(1,13):
        s = f"{yr}-{mo:02d}-01"
        ed = ee.Date(s).advance(1,"month")
        lmo = landsat_col.filterDate(s, ed).select(["LST","NDVI","NDBI","MNDWI"])
        ln = lmo.size().getInfo()
        cmo = get_coarse_composite(year=yr, month=mo)
        cv = cmo.reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi,scale=1000,maxPixels=1e9).get("LST_coarse").getInfo()
        if ln==0 and cv is None: continue
        if ln>=1 and cv is not None:
            fused = fuse_lst(lmo.median().clip(aoi_buffer), cmo, aoi_buffer, f"{yr}-{mo:02d}")
            method = f"Fused ({coarse_source_used})"
        elif ln>=1:
            fused = lmo.select("LST").median().clip(aoi_buffer).rename("LST_fused")
            method = "Landsat only"
        else:
            fused = cmo.select("LST_coarse").resample("bilinear").reproject(crs="EPSG:4326",scale=30).rename("LST_fused")
            method = f"{coarse_source_used} resampled"
        mv = fused.reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi,scale=30,maxPixels=1e9).get("LST_fused").getInfo()
        monthly_results.append({"Year":yr,"Month":mo,"Date":f"{yr}-{mo:02d}","L_scenes":ln,"Method":method,
                                "Mean_LST":round(mv,2) if mv else None})
        if mv: print(f"  {yr}-{mo:02d} L={ln} {method} {mv:.1f}C")
monthly_df = pd.DataFrame(monthly_results)
print(f"\n{len(monthly_df)} months.")


In [ ]:
# 5E. MONTHLY PLOT
fig,ax=plt.subplots(figsize=(14,5))
bc={"Fused (S3)":"#d73027","Fused (MODIS)":"#d73027","Landsat only":"#4575b4","S3 resampled":"#fdae61","MODIS resampled":"#fdae61"}
for method,grp in monthly_df.groupby("Method"):
    ax.plot(grp["Date"],grp["Mean_LST"],"o-",color=bc.get(method,"gray"),label=method,ms=4,lw=1)
# Add threshold lines
ax.axhline(y=LST_HIGH, color="#d73027", ls="--", lw=1.5, alpha=0.7, label=f"High threshold ({LST_HIGH}C)")
ax.axhline(y=LST_MODERATE, color="#fdae61", ls="--", lw=1.5, alpha=0.7, label=f"Moderate threshold ({LST_MODERATE}C)")
ax.set_xlabel("Month");ax.set_ylabel("LST (C)");ax.set_title(f"{PROJECT_NAME} — Monthly LST with Thresholds")
ax.legend(fontsize=9);ax.set_xticks(ax.get_xticks()[::3]);plt.xticks(rotation=45)
plt.tight_layout();plt.savefig("monthly_lst.png",dpi=150,bbox_inches="tight");plt.show()


## 5F. Fusion Validation

In [ ]:
# 5F. VALIDATION
vs = annual_composite.select("LST").rename("L").addBands(annual_composite.select("LST_fused").rename("F"))
vd = pd.DataFrame([f["properties"] for f in vs.sample(region=aoi,scale=30,numPixels=1000,seed=42,geometries=False).getInfo()["features"]]).dropna()
fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle(f"{PROJECT_NAME} — Fusion Validation ({coarse_source_used})",fontsize=14,fontweight="bold")
axes[0].scatter(vd["L"],vd["F"],alpha=0.3,s=8,c="#d73027");axes[0].plot([vd.min().min(),vd.max().max()],[vd.min().min(),vd.max().max()],"k--")
axes[0].set_xlabel("Landsat");axes[0].set_ylabel("Fused");axes[0].set_title("Scatter")
axes[1].hist(vd["L"],bins=40,alpha=0.6,color="#4575b4",label="Landsat");axes[1].hist(vd["F"],bins=40,alpha=0.6,color="#d73027",label="Fused");axes[1].legend();axes[1].set_title("Distribution")
d=vd["F"]-vd["L"];axes[2].hist(d,bins=40,color="#fdae61",alpha=0.7);axes[2].axvline(0,color="k",ls="--")
axes[2].set_title(f"Residual (mean={d.mean():.2f})");plt.tight_layout();plt.savefig("fusion_val.png",dpi=150);plt.show()


---
## 6. LST Threshold Analysis — Extreme Heat / Moderate / Cool Days

Classifies every month by its mean fused LST into:
- **Extreme Heat**: mean LST ≥ threshold (default 35°C)
- **Moderate Heat**: mean LST ≥ moderate threshold and < high (default 30–35°C)
- **Cool/Normal**: mean LST < moderate threshold

Also computes percentile distributions, exceedance counts, and heat stress indices.


In [ ]:
# 6A. THRESHOLD CLASSIFICATION
valid = monthly_df.dropna(subset=["Mean_LST"]).copy()

def classify_lst(val):
    if val >= LST_HIGH: return "Extreme Heat"
    elif val >= LST_MODERATE: return "Moderate Heat"
    else: return "Cool/Normal"

valid["Category"] = valid["Mean_LST"].apply(classify_lst)

# Season assignment
def get_season(mo):
    if mo in DRY_MONTHS: return "Dry"
    return "Wet"
valid["Season"] = valid["Month"].apply(get_season)

# Summary counts
cat_counts = valid["Category"].value_counts()
print("LST Category Distribution")
print("=" * 40)
for cat in ["Extreme Heat", "Moderate Heat", "Cool/Normal"]:
    n = cat_counts.get(cat, 0)
    pct = n / len(valid) * 100
    print(f"  {cat:<16s}: {n:3d} months ({pct:.1f}%)")

print(f"\nThresholds: High >= {LST_HIGH}C | Moderate >= {LST_MODERATE}C")
print(f"Total months analyzed: {len(valid)}")


In [ ]:
# 6B. PERCENTILE ANALYSIS
percentiles = valid["Mean_LST"].describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
print("LST Percentile Distribution")
print("=" * 40)
print(percentiles.round(2).to_string())

# Percentile by season
print("\nPercentiles by Season:")
print("-" * 50)
for season in ["Dry", "Wet"]:
    grp = valid[valid["Season"] == season]["Mean_LST"]
    if len(grp) > 0:
        print(f"  {season}: P10={grp.quantile(0.10):.1f} | P25={grp.quantile(0.25):.1f} | "
              f"P50={grp.quantile(0.50):.1f} | P75={grp.quantile(0.75):.1f} | P90={grp.quantile(0.90):.1f}")


In [ ]:
# 6C. THRESHOLD EXCEEDANCE PLOT
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"{PROJECT_NAME} — LST Threshold Analysis", fontsize=14, fontweight="bold")

# (1) Category bar chart
cats = ["Extreme Heat", "Moderate Heat", "Cool/Normal"]
colors_cat = {"Extreme Heat": "#d73027", "Moderate Heat": "#fdae61", "Cool/Normal": "#4575b4"}
counts = [cat_counts.get(c, 0) for c in cats]
axes[0,0].bar(cats, counts, color=[colors_cat[c] for c in cats], edgecolor="white")
axes[0,0].set_ylabel("Number of Months"); axes[0,0].set_title("Monthly LST Category Distribution")
for i, (cat, cnt) in enumerate(zip(cats, counts)):
    axes[0,0].text(i, cnt + 0.3, str(cnt), ha="center", fontweight="bold")

# (2) Monthly LST with color-coded categories
for _, row in valid.iterrows():
    color = colors_cat[row["Category"]]
    axes[0,1].bar(row["Date"], row["Mean_LST"], color=color, width=0.8)
axes[0,1].axhline(LST_HIGH, color="#d73027", ls="--", lw=1, alpha=0.7)
axes[0,1].axhline(LST_MODERATE, color="#fdae61", ls="--", lw=1, alpha=0.7)
axes[0,1].set_ylabel("Mean LST (C)"); axes[0,1].set_title("Monthly LST (colored by category)")
axes[0,1].set_xticks(axes[0,1].get_xticks()[::6]); axes[0,1].tick_params(axis="x", rotation=45)

# (3) Box plot by season
valid.boxplot(column="Mean_LST", by="Season", ax=axes[1,0], patch_artist=True,
              boxprops=dict(facecolor="#fee090"), medianprops=dict(color="#d73027", lw=2))
axes[1,0].axhline(LST_HIGH, color="#d73027", ls="--", lw=1, alpha=0.7, label=f"High ({LST_HIGH}C)")
axes[1,0].axhline(LST_MODERATE, color="#fdae61", ls="--", lw=1, alpha=0.7, label=f"Moderate ({LST_MODERATE}C)")
axes[1,0].set_ylabel("Mean LST (C)"); axes[1,0].set_title("Seasonal Distribution")
axes[1,0].legend(fontsize=8); axes[1,0].set_xlabel("")
plt.sca(axes[1,0]); plt.title("Seasonal Distribution")

# (4) Year-by-year exceedance
yearly_exc = valid.groupby("Year").apply(
    lambda g: pd.Series({
        "High": (g["Mean_LST"] >= LST_HIGH).sum(),
        "Moderate": ((g["Mean_LST"] >= LST_MODERATE) & (g["Mean_LST"] < LST_HIGH)).sum(),
        "Cool": (g["Mean_LST"] < LST_MODERATE).sum()
    })).reset_index()
x = np.arange(len(yearly_exc))
w = 0.25
axes[1,1].bar(x - w, yearly_exc["High"], w, color="#d73027", label="High")
axes[1,1].bar(x, yearly_exc["Moderate"], w, color="#fdae61", label="Moderate")
axes[1,1].bar(x + w, yearly_exc["Cool"], w, color="#4575b4", label="Cool")
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(yearly_exc["Year"].astype(int))
axes[1,1].set_ylabel("Months"); axes[1,1].set_title("Yearly Threshold Exceedance")
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("lst_thresholds.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 6D. HEAT STRESS SUMMARY TABLE
summary_rows = []
for yr in sorted(valid["Year"].unique()):
    ydf = valid[valid["Year"] == yr]
    for season in ["Dry", "Wet"]:
        sdf = ydf[ydf["Season"] == season]
        if len(sdf) == 0: continue
        summary_rows.append({
            "Year": int(yr), "Season": season,
            "Months": len(sdf),
            "Mean LST": round(sdf["Mean_LST"].mean(), 2),
            "Max LST": round(sdf["Mean_LST"].max(), 2),
            "Min LST": round(sdf["Mean_LST"].min(), 2),
            "High Days": int((sdf["Mean_LST"] >= LST_HIGH).sum()),
            "Moderate Days": int(((sdf["Mean_LST"] >= LST_MODERATE) & (sdf["Mean_LST"] < LST_HIGH)).sum()),
            "Cool Days": int((sdf["Mean_LST"] < LST_MODERATE).sum())
        })

summary_df = pd.DataFrame(summary_rows)
print("Heat Stress Summary (by Year x Season)")
print("=" * 95)
print(summary_df.to_string(index=False))
summary_df.to_csv("heat_stress_summary.csv", index=False)
print("\nSaved: heat_stress_summary.csv")


In [ ]:
# 6E. PIXEL-LEVEL HEAT CLASSIFICATION MAP
# Classify each 30m pixel by its annual mean fused LST
lst_fused = annual_composite.select("LST_fused")

extreme_mask = lst_fused.gte(LST_HIGH).selfMask().rename("extreme_heat")
moderate_mask = lst_fused.gte(LST_MODERATE).And(lst_fused.lt(LST_HIGH)).selfMask().rename("moderate_heat")
cool_mask = lst_fused.lt(LST_MODERATE).selfMask().rename("cool_normal")

# Area calculations
pixel_area = ee.Image.pixelArea()
extreme_area = extreme_mask.multiply(pixel_area).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e9).get("extreme_heat").getInfo()
moderate_area = moderate_mask.multiply(pixel_area).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e9).get("moderate_heat").getInfo()
cool_area = cool_mask.multiply(pixel_area).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=30, maxPixels=1e9).get("cool_normal").getInfo()

total = (extreme_area or 0) + (moderate_area or 0) + (cool_area or 0)
print("Pixel-Level Heat Classification (Annual)")
print("=" * 50)
if total > 0:
    print(f"  Extreme Heat (>={LST_HIGH}C): {(extreme_area or 0)/1e6:.3f} km2 ({(extreme_area or 0)/total*100:.1f}%)")
    print(f"  Moderate Heat ({LST_MODERATE}-{LST_HIGH}C): {(moderate_area or 0)/1e6:.3f} km2 ({(moderate_area or 0)/total*100:.1f}%)")
    print(f"  Cool/Normal (<{LST_MODERATE}C): {(cool_area or 0)/1e6:.3f} km2 ({(cool_area or 0)/total*100:.1f}%)")

# Map
m_heat = make_map(center_lat, center_lon, 14)
add_ee_layer(m_heat, extreme_mask.clip(aoi), {"min":0,"max":1,"palette":["d73027"]}, f"Extreme (>={LST_HIGH}C)")
add_ee_layer(m_heat, moderate_mask.clip(aoi), {"min":0,"max":1,"palette":["fdae61"]}, f"Moderate ({LST_MODERATE}-{LST_HIGH}C)")
add_ee_layer(m_heat, cool_mask.clip(aoi), {"min":0,"max":1,"palette":["4575b4"]}, f"Cool (<{LST_MODERATE}C)", shown=False)
folium.LayerControl().add_to(m_heat)
m_heat


## 7. UHI Intensity

In [ ]:
# 7. UHI
def compute_uhi(comp, band, label):
    lst=comp.select(band)
    ref=ee.Number(lst.reduceRegion(reducer=ee.Reducer.mean(),geometry=aoi_ring,scale=30,maxPixels=1e9).get(band))
    print(f"  {label} ref:{ref.getInfo():.2f}C")
    return lst.subtract(ref).rename("UHI_intensity")
print("Fused UHI:")
uhi_dry=compute_uhi(dry_composite,"LST_fused","Dry")
uhi_wet=compute_uhi(wet_composite,"LST_fused","Wet")
uhi_annual=compute_uhi(annual_composite,"LST_fused","Annual")
print("\nLandsat UHI:")
uhi_dry_l=compute_uhi(dry_composite,"LST","Dry")
uhi_wet_l=compute_uhi(wet_composite,"LST","Wet")
uhi_annual_l=compute_uhi(annual_composite,"LST","Annual")


## 8. Maps

In [ ]:
# 8. MAP
lv={"min":25,"max":45,"palette":["313695","4575b4","74add1","abd9e9","e0f3f8","ffffbf","fee090","fdae61","f46d43","d73027","a50026"]}
uv={"min":-3,"max":6,"palette":["2166ac","67a9cf","d1e5f0","fddbc7","ef8a62","b2182b"]}
m2=make_map(center_lat,center_lon,14)
add_ee_layer(m2,dry_composite.select("LST_fused").clip(aoi),lv,"Fused LST Dry")
add_ee_layer(m2,wet_composite.select("LST_fused").clip(aoi),lv,"Fused LST Wet",False)
add_ee_layer(m2,annual_composite.select("LST").clip(aoi),lv,"Landsat LST",False)
add_ee_layer(m2,uhi_dry.clip(aoi),uv,"UHI Dry")
add_ee_layer(m2,uhi_wet.clip(aoi),uv,"UHI Wet",False)
add_ee_layer(m2,annual_composite.select("NDVI").clip(aoi),
    {"min":-0.1,"max":0.6,"palette":["FFFFFF","CE7E45","DF923D","F1B555","FCD163","99B718","74A901","66A000","529400","3E8601","207401","056201"]},"NDVI",False)
folium.LayerControl().add_to(m2)
m2


---
## 9. Ancillary Data

In [ ]:
# 9. ANCILLARY
pop=(ee.ImageCollection("WorldPop/GP/100m/pop").filterBounds(aoi).sort("system:time_start",False).first().select("population").clip(aoi))
print(f"Pop: WorldPop {ee.Date(pop.get('system:time_start')).format('YYYY').getInfo()}")
ghsl=ee.Image("JRC/GHSL/P2023A/GHS_BUILT_S/2020").select("built_surface").clip(aoi);print("GHSL")
esa_lc=ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi);print("ESA WorldCover")
health_dist=None
try:
    ha=ee.FeatureCollection("projects/sat-io/open-datasets/health-site-node").filterBounds(aoi).merge(ee.FeatureCollection("projects/sat-io/open-datasets/health-site-way").filterBounds(aoi))
    hc=ha.size().getInfo()
    if hc>0: health_dist=ha.distance(5000).clip(aoi).rename("health_dist");print(f"Healthsites:{hc}")
    else: raise Exception("0")
except: health_dist=ee.Image.constant(0).rename("health_dist").clip(aoi).float();print("Health: placeholder")
try:
    ob=ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons").filterBounds(aoi)
    bldg_raster=ob.reduceToImage(properties=["confidence"],reducer=ee.Reducer.count()).rename("building_count").clip(aoi)
    print(f"OpenBldgs:{ob.size().getInfo()}")
except: bldg_raster=ghsl.rename("building_count");print("Bldgs: GHSL fallback")


---
## 10. Statistical Analysis

In [ ]:
# 10A. ZONAL STATS
def zs(img,band,geom,sc=30):
    return img.select(band).reduceRegion(reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(),sharedInputs=True).combine(ee.Reducer.stdDev(),sharedInputs=True).combine(ee.Reducer.percentile([10,25,50,75,90]),sharedInputs=True),geometry=geom,scale=sc,maxPixels=1e9).getInfo()

rows=[]
for sn,comp in [("Ann",annual_composite),("Dry",dry_composite),("Wet",wet_composite)]:
    for b,lb in [("LST_fused","Fused"),("LST","Landsat")]:
        s=zs(comp,b,aoi)
        rows.append({"Season":sn,"Src":lb,
            "Mean":round(s.get(f"{b}_mean",0),2),
            "Min":round(s.get(f"{b}_min",0),2),
            "P10":round(s.get(f"{b}_p10",0),2),
            "P25":round(s.get(f"{b}_p25",0),2),
            "Median":round(s.get(f"{b}_p50",0),2),
            "P75":round(s.get(f"{b}_p75",0),2),
            "P90":round(s.get(f"{b}_p90",0),2),
            "Max":round(s.get(f"{b}_max",0),2),
            "Std":round(s.get(f"{b}_stdDev",0),2)})
stats_df=pd.DataFrame(rows)
print("LST Summary — Fused vs Landsat (with percentiles)")
print("=" * 110)
print(stats_df.to_string(index=False))


In [ ]:
# 10B. SAMPLING + CORRELATION
stk=annual_composite.select("LST_fused").rename("LST").addBands(annual_composite.select("NDVI")).addBands(annual_composite.select("NDBI")).addBands(annual_composite.select("MNDWI")).addBands(pop.rename("POP")).addBands(ghsl.rename("BUILT"))
df=pd.DataFrame([f["properties"] for f in stk.sample(region=aoi,scale=30,numPixels=2000,seed=42,geometries=False).getInfo()["features"]]).dropna()
print(f"{len(df)} pixels")

fig,axes=plt.subplots(2,3,figsize=(16,10));fig.suptitle(f"{PROJECT_NAME} — Correlation",fontsize=14)
corr=df[["LST","NDVI","NDBI","MNDWI","POP","BUILT"]].corr()
sns.heatmap(corr,annot=True,cmap="RdBu_r",center=0,ax=axes[0,0],fmt=".2f")
for ax,col,cc,t in [(axes[0,1],"NDVI","green","NDVI"),(axes[0,2],"NDBI","orange","NDBI"),(axes[1,0],"POP","purple","Pop"),(axes[1,1],"BUILT","brown","Built")]:
    ax.scatter(df[col],df["LST"],alpha=0.3,s=5,c=cc);ax.set_xlabel(col);ax.set_ylabel("LST");ax.set_title(f"LST vs {t}")
axes[1,2].hist(df["LST"],bins=40,color="#d73027",alpha=0.7);axes[1,2].set_title("LST Dist")
axes[1,2].axvline(LST_HIGH,color="black",ls="--",lw=1.5,label=f"High ({LST_HIGH}C)")
axes[1,2].axvline(LST_MODERATE,color="gray",ls="--",lw=1.5,label=f"Mod ({LST_MODERATE}C)")
axes[1,2].legend(fontsize=8)
plt.tight_layout();plt.savefig("correlation.png",dpi=150);plt.show()


## 11. Temporal Trends

In [ ]:
# 11. TRENDS
fig,axes=plt.subplots(2,1,figsize=(14,9),gridspec_kw={"height_ratios":[2,1]})
v=monthly_df.dropna(subset=["Mean_LST"])
axes[0].plot(v["Date"],v["Mean_LST"],"o-",color="#d73027",ms=3,lw=1)
axes[0].axhline(LST_HIGH,color="#d73027",ls="--",lw=1,alpha=0.5)
axes[0].axhline(LST_MODERATE,color="#fdae61",ls="--",lw=1,alpha=0.5)
axes[0].set_ylabel("LST (C)");axes[0].set_title(f"{PROJECT_NAME} — Monthly")
axes[0].set_xticks(axes[0].get_xticks()[::6]);axes[0].tick_params(axis="x",rotation=45)
y=v.groupby("Year").agg(M=("Mean_LST","mean"),N=("Mean_LST","count")).reset_index()
axes[1].bar(y["Year"].astype(str),y["M"],color="#fc8d59")
for _,r in y.iterrows(): axes[1].annotate(f"{r['M']:.1f}",(str(int(r["Year"])),r["M"]),ha="center",va="bottom",fontsize=9)
axes[1].set_ylabel("LST");axes[1].set_title("Yearly")
plt.tight_layout();plt.savefig("trends.png",dpi=150);plt.show()


---
## 12. Vulnerability Index

In [ ]:
# 12. VULNERABILITY
lst_norm=min_max_normalize(annual_composite,aoi,"LST_fused").rename("LST_norm")
pop_norm=min_max_normalize(pop,aoi).rename("POP_norm")
built_norm=min_max_normalize(ghsl,aoi).rename("BUILT_norm")
ndvi_inv=ee.Image.constant(1).subtract(min_max_normalize(annual_composite,aoi,"NDVI")).rename("NDVI_inv")
health_norm=min_max_normalize(health_dist,aoi).rename("HEALTH_norm")
vulnerability=(lst_norm.multiply(W_LST).add(pop_norm.multiply(W_POP)).add(built_norm.multiply(W_BUILD)).add(ndvi_inv.multiply(W_NDVI)).add(health_norm.multiply(W_HEALTH))).rename("vulnerability").clip(aoi)
print(f"Vulnerability computed. W: LST={W_LST} POP={W_POP} BUILD={W_BUILD} NDVI={W_NDVI} HEALTH={W_HEALTH}")


In [ ]:
# 12B. VULN MAP
vv={"min":0,"max":1,"palette":["1a9850","91cf60","d9ef8b","ffffbf","fee08b","fc8d59","d73027"]}
m3=make_map(center_lat,center_lon,14)
add_ee_layer(m3,vulnerability,vv,"Vulnerability")
add_ee_layer(m3,lst_norm.clip(aoi),{"min":0,"max":1,"palette":["blue","yellow","red"]},"LST",False)
add_ee_layer(m3,pop_norm.clip(aoi),{"min":0,"max":1,"palette":["white","purple"]},"Pop",False)
add_ee_layer(m3,ndvi_inv.clip(aoi),{"min":0,"max":1,"palette":["green","white","brown"]},"VegDeficit",False)
folium.LayerControl().add_to(m3)
m3


## 13. Barangay Ranking

In [ ]:
# 13. BARANGAY
try:
    bg=ee.FeatureCollection("projects/sat-io/open-datasets/geoboundaries/CGAZ_ADM4").filterBounds(aoi)
    if bg.size().getInfo()==0: raise Exception("0")
    bv=vulnerability.reduceRegions(collection=bg,reducer=ee.Reducer.mean().setOutputs(["vm"]),scale=30)
    bl=annual_composite.select("LST_fused").reduceRegions(collection=bg,reducer=ee.Reducer.mean().setOutputs(["lm"]),scale=30)
    rk=[{"Barangay":v["properties"].get("shapeName","?"),"Vuln":round(v["properties"].get("vm",0),3),"LST":round(l["properties"].get("lm",0),2)} for v,l in zip(bv.getInfo()["features"],bl.getInfo()["features"])]
    rank_df=pd.DataFrame(rk).sort_values("Vuln",ascending=False);print(rank_df.to_string(index=False))
except Exception as e: print(f"Barangay N/A: {e}")


---
## 14. Export

In [ ]:
# 14. EXPORT
ep=dict(region=aoi,scale=30,maxPixels=1e9,crs="EPSG:4326")
for img,nm in [(vulnerability.toFloat(),"vulnerability"),(annual_composite.select("LST_fused").toFloat(),"LST_fused"),
               (annual_composite.select("LST").toFloat(),"LST_landsat"),(uhi_dry.toFloat(),"UHI_dry"),
               (uhi_annual.toFloat(),"UHI_annual"),(extreme_mask.unmask(0).toFloat(),"extreme_heat_mask")]:
    ee.batch.Export.image.toDrive(image=img,description=f"{PROJECT_NAME}_{nm}",folder=PROJECT_NAME,**ep).start()
    print(f"Export: {nm}")
print(f"\nhttps://code.earthengine.google.com/tasks")
stats_df.to_csv("stats.csv",index=False);monthly_df.to_csv("monthly.csv",index=False)
summary_df.to_csv("heat_stress_summary.csv",index=False)
if "rank_df" in dir(): rank_df.to_csv("ranking.csv",index=False)
print("CSVs saved.")


---
## 15. Cooling Sites

In [ ]:
# 15. COOLING
cool=esa_lc.eq(10).Or(esa_lc.eq(20)).Or(esa_lc.eq(30)).selfMask().rename("cool")
vp75=vulnerability.reduceRegion(reducer=ee.Reducer.percentile([75]),geometry=aoi,scale=30,maxPixels=1e9).get("vulnerability")
hv=vulnerability.gte(ee.Number(vp75)).selfMask()
m4=make_map(center_lat,center_lon,14)
add_ee_layer(m4,vulnerability,vv,"Vulnerability")
add_ee_layer(m4,hv.clip(aoi),{"min":0,"max":1,"palette":["red"]},"High Vuln")
add_ee_layer(m4,cool.clip(aoi),{"min":0,"max":1,"palette":["00ff00"]},"Cool Spaces",False)
folium.LayerControl().add_to(m4)
m4


---
## Summary (v4)

### Outputs:
1. Landsat 8+9 (30 m, ~8-day) + S3 SLSTR / MODIS (1 km, daily)
2. Regression-based LST fusion → 30 m with residual correction
3. Monthly fused LST time-series
4. **LST threshold analysis**: extreme heat / moderate / cool classification
5. **Pixel-level heat map**: area-weighted heat zone classification
6. **Heat stress summary table**: by year × season
7. **Percentile distributions**: P10/P25/P50/P75/P90 by season
8. UHI intensity (fused + Landsat comparison)
9. Vulnerability index + barangay ranking
10. Exported GeoTIFFs + CSVs

### S3 speed improvement:
Downloads only `LST_in.nc` (~1 MB) + `geodetic_in.nc` (~1 MB) per scene via CDSE Nodes API,
instead of the full product zip (~100-200 MB). ~50x faster.

### Change AOI: edit `AOI_BBOX` in config → re-run all

---
*MSF · Z_GIS Salzburg*
